In [1]:
import os
from dotenv import load_dotenv

# Import LangSmith tracing (debugging in LangSmith.com)
from langsmith import Client # type: ignore

# Langgraph packages
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_tableau.tools.simple_datasource_qa import initialize_simple_datasource_qa

from utilities.chat import print_stream
from utilities.prompt import AGENT_SYSTEM_PROMPT

# Load environment
load_dotenv()

# Add LangSmith tracing
langsmith_client = Client()

config = {
    "run_name": "Tableau Langchain Main.py"
}

# Initialize the Tableau data source tool
analyze_datasource = initialize_simple_datasource_qa(
    domain=os.environ['TABLEAU_DOMAIN'],
    site=os.environ['TABLEAU_SITE'],
    jwt_client_id=os.environ['TABLEAU_JWT_CLIENT_ID'],
    jwt_secret_id=os.environ['TABLEAU_JWT_SECRET_ID'],
    jwt_secret=os.environ['TABLEAU_JWT_SECRET'],
    tableau_api_version=os.environ['TABLEAU_API_VERSION'],
    tableau_user=os.environ['TABLEAU_USER'],
    datasource_luid=os.environ['DATASOURCE_LUID'],
    tooling_llm_model="gpt-4.1-nano",
    model_provider="openai"
)

In [2]:
os.environ['DATASOURCE_LUID']

'6168364c-9e26-4b88-9400-c577ec99c40b'

In [3]:
# load the List of Tools to be used by the Agent. In this case we will just load our data source Q&A tool.
tools = [analyze_datasource]

In [4]:
response = analyze_datasource.invoke({
    "user_input": "What is the sum of nuance_net_sales?"
})
print(response)


ERROR:root:Unexpected error in get_headlessbi_data: Failed to query data source via Tableau VizQL Data Service. Status code: 400. Response: {"errorCode":"400803","message":"Unknown Field: nuance_net_sales.","datetime":"2025-06-19T14:31:36.408280353Z"}


ToolException: 
                Tableau's VizQL Data Service return an error for the generated query:

                {"fields": [{"fieldCaption": "nuance_net_sales", "function": "SUM"}]}

                The user_input used to write this query was:

                What is the sum of nuance_net_sales?

                This was the error:

                An unexpected error occurred: Failed to query data source via Tableau VizQL Data Service. Status code: 400. Response: {"errorCode":"400803","message":"Unknown Field: nuance_net_sales.","datetime":"2025-06-19T14:31:36.408280353Z"}

                Consider retrying this tool with the same inputs but include the previous query
                causing the error and the error itself for the tool to correct itself on a retry.
                If the error was an empty array, this usually indicates an incorrect filter value
                was applied, thus returning no data
                

In [16]:
import os
from dotenv import load_dotenv

# Import LangSmith tracing (debugging in LangSmith.com)
from langsmith import Client # type: ignore

# Langgraph packages
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_tableau.tools.simple_datasource_qa import initialize_simple_datasource_qa

from utilities.chat import print_stream
from utilities.prompt import AGENT_SYSTEM_PROMPT
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType




# Load environment
load_dotenv()

# Add LangSmith tracing
langsmith_client = Client()

config = {
    "run_name": "Tableau Langchain Main.py"
}

# Initialize the Tableau data source tool
analyze_datasource = initialize_simple_datasource_qa(
    domain=os.environ['TABLEAU_DOMAIN'],
    site=os.environ['TABLEAU_SITE'],
    jwt_client_id=os.environ['TABLEAU_JWT_CLIENT_ID'],
    jwt_secret_id=os.environ['TABLEAU_JWT_SECRET_ID'],
    jwt_secret=os.environ['TABLEAU_JWT_SECRET'],
    tableau_api_version=os.environ['TABLEAU_API_VERSION'],
    tableau_user=os.environ['TABLEAU_USER'],
    datasource_luid=os.environ['DATASOURCE_LUID'],
    tooling_llm_model="gpt-4.1-nano",
    model_provider="openai"
)

analyze_datasource.name = "query_tableau_datasource"
analyze_datasource.description = (
    "Use this tool to query the Tableau datasource for things like who owns a task, due dates, "
    "priority, progress, or anything related to project management data."
)


# Create the agent
llm = ChatOpenAI(model="gpt-4.1", temperature=0)
tools = [analyze_datasource]

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


TableauLangChain = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    memory=memory,
    verbose=True
)

In [ ]:
# Usage
your_prompt = 'who is responbible for Tele Comps?'

# Run the agent
messages = {"input": your_prompt}
print_stream(TableauLangChain.stream(messages, config=config))





> Entering new Tableau Langchain Main.py chain...
================================== Ai Message ==================================

Invoking: `query_tableau_datasource` with `{'user_input': "Who is responsible for the task or project named 'Tele Comps'?", 'previous_call_error': None, 'previous_vds_payload': None}`




In [15]:
# Follow-up without repeating context
response2 = TableauLangChain.run("Tele Comps means a task name in the project sheet.")
print(response2)

C:\Users\227457\AppData\Local\Temp\ipykernel_1796\4284367074.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response2 = TableauLangChain.run("Tele Comps means a task name in the project sheet.")




> Entering new AgentExecutor chain...
Thank you for clarifying! So, in your project sheet, "Tele Comps" refers to a specific task name. If you need to filter, analyze, or report on data related to the "Tele Comps" task, just let me know what you’d like to do (for example: "Show me the status of all 'Tele Comps' tasks," or "Summarize hours spent on 'Tele Comps' this month"), and I can help you with that.

If you have any specific questions or need to perform an action related to "Tele Comps," please provide more details!

> Finished chain.
Thank you for clarifying! So, in your project sheet, "Tele Comps" refers to a specific task name. If you need to filter, analyze, or report on data related to the "Tele Comps" task, just let me know what you’d like to do (for example: "Show me the status of all 'Tele Comps' tasks," or "Summarize hours spent on 'Tele Comps' this month"), and I can help you with that.

If you have any specific questions or need to perform an action related to "Tele Co

In [37]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_tableau.tools.query_tool import TableauDatasourceQueryTool

# Load .env variables
load_dotenv()

# Set up the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Create query tool for live Tableau querying
query_tool = TableauDatasourceQueryTool(
    domain=os.environ['TABLEAU_DOMAIN'],
    site=os.environ['TABLEAU_SITE'],
    jwt_client_id=os.environ['TABLEAU_JWT_CLIENT_ID'],
    jwt_secret_id=os.environ['TABLEAU_JWT_SECRET_ID'],
    jwt_secret=os.environ['TABLEAU_JWT_SECRET'],
    tableau_api_version=os.environ['TABLEAU_API_VERSION'],
    tableau_user=os.environ['TABLEAU_USER'],
    datasource_luid=os.environ['DATASOURCE_LUID'],  # 🟢 Your confirmed good LUID
    llm=llm,
)

# Run a direct query (you can change the prompt)
response = query_tool.invoke({
    "query": "Show me the top 5 stores by sales."
})

print(response)


ModuleNotFoundError: No module named 'langchain_tableau.tools.query_tool'

In [38]:
from langchain_tableau.tools.query_tool import TableauDatasourceQueryTool


ModuleNotFoundError: No module named 'langchain_tableau.tools.query_tool'

In [36]:
!pip install --upgrade langchain-tableau


In [7]:
from IPython.display import Markdown, display

model = ChatOpenAI(model="gpt-4o", temperature=0)

tableauAgent = create_react_agent(model, tools)

# Run the agent
messages = tableauAgent.invoke(
    {
        "messages": [
            (
                "human",
                "what are the column names in the rlb_metrics_temp datasource?",
            )
        ]
    }
)
messages
# display(Markdown(messages['messages'][3].content)) #display a nicely formatted answer for successful generations

{'messages': [HumanMessage(content='what are the column names in the rlb_metrics_temp datasource?', additional_kwargs={}, response_metadata={}, id='4a4fa209-94ad-480e-bf80-512e3d0004a9'),
  AIMessage(content='I currently don\'t have access to the specific schema or column names of the "rlb_metrics_temp" datasource. You would need to refer to the documentation or schema details of that datasource in your Tableau environment to get the column names. If you have access to Tableau, you can also explore the datasource directly to view its structure.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 477, 'total_tokens': 543, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_07871e2ad8', 'id': 'chatcmpl-BjoHbqOjDEs

In [25]:
response = tools[0].invoke({"user_input": "What are the column names in the datasource?"})
print(response)


IndexError: list index out of range

In [22]:
from langchain_tableau.tools.simple_datasource_qa import initialize_simple_datasource_qa

analyze_tool = initialize_simple_datasource_qa(
    domain=os.environ['TABLEAU_DOMAIN'],
    site=os.environ['TABLEAU_SITE'],
    jwt_client_id=os.environ['TABLEAU_JWT_CLIENT_ID'],
    jwt_secret_id=os.environ['TABLEAU_JWT_SECRET_ID'],
    jwt_secret=os.environ['TABLEAU_JWT_SECRET'],
    tableau_api_version=os.environ['TABLEAU_API_VERSION'],
    tableau_user=os.environ['TABLEAU_USER'],
    tooling_llm_model="gpt-4.1-nano",
    model_provider="openai"
)

response = analyze_tool.invoke({"user_input": "List all available datasources."})
print(response)


IndexError: list index out of range

In [26]:
analyze_tool = initialize_simple_datasource_qa(
    domain=os.environ['TABLEAU_DOMAIN'],
    site=os.environ['TABLEAU_SITE'],
    jwt_client_id=os.environ['TABLEAU_JWT_CLIENT_ID'],
    jwt_secret_id=os.environ['TABLEAU_JWT_SECRET_ID'],
    jwt_secret=os.environ['TABLEAU_JWT_SECRET'],
    tableau_api_version=os.environ['TABLEAU_API_VERSION'],
    tableau_user=os.environ['TABLEAU_USER'],
    datasource_name="rlb_metrics_temp",  # 👈 Use name instead
    tooling_llm_model="gpt-4.1-nano",
    model_provider="openai"
)


TypeError: initialize_simple_datasource_qa() got an unexpected keyword argument 'datasource_name'. Did you mean 'datasource_luid'?

In [27]:
# After authenticating...
ds_response = requests.get(
    f"{os.environ['TABLEAU_DOMAIN']}/api/{os.environ['TABLEAU_API_VERSION']}/sites/{site_id}/datasources",
    headers={"X-Tableau-Auth": token}
)
print(ds_response.json())


NameError: name 'site_id' is not defined

In [29]:
import os, jwt, requests
from datetime import datetime, timedelta, timezone

# Step 1: Create JWT token
def create_jwt():
    now = datetime.now(timezone.utc)
    payload = {
        "iss": os.environ['TABLEAU_JWT_CLIENT_ID'],
        "sub": os.environ['TABLEAU_USER'],  # Must match case exactly (e.g., JWang@...)
        "aud": "tableau",
        "exp": int((now + timedelta(minutes=5)).timestamp()),
        "iat": int(now.timestamp()),
        "jti": os.urandom(16).hex()
    }
    return jwt.encode(payload, os.environ['TABLEAU_JWT_SECRET'], algorithm="HS256")

# Step 2: Sign in to Tableau and get token
jwt_token = create_jwt()
auth_url = f"{os.environ['TABLEAU_DOMAIN']}/api/{os.environ['TABLEAU_API_VERSION']}/auth/signin"

auth_response = requests.post(
    auth_url,
    json={
        "credentials": {
            "jwt": jwt_token,
            "site": {"contentUrl": os.environ['TABLEAU_SITE']}
        }
    },
    headers={"Content-Type": "application/json"}
)

print("Auth Status Code:", auth_response.status_code)
print("Auth Response:", auth_response.text)

# Exit early if authentication fails
if auth_response.status_code != 200:
    raise Exception("Authentication failed. Check JWT config or Tableau user access.")

auth_data = auth_response.json()
token = auth_data['credentials']['token']
site_id = auth_data['credentials']


Auth Status Code: 401
Auth Response: <?xml version='1.0' encoding='UTF-8'?><tsResponse xmlns="http://tableau.com/api" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://tableau.com/api https://help.tableau.com/samples/en-us/rest_api/ts-api_3_25.xsd"><error code="401001"><summary>Signin Error</summary><detail>Error signing in to Tableau Server (10083)</detail></error></tsResponse>


Exception: Authentication failed. Check JWT config or Tableau user access.

In [42]:
!pip install --upgrade langchain-tableau


In [43]:
from langchain_tableau.tools.query_tool import TableauDatasourceQueryTool

ModuleNotFoundError: No module named 'langchain_tableau.tools.query_tool'

In [44]:
datasources = response.json().get('data', {}).get('publishedDatasources', [])
if not datasources:
    raise ValueError("No metadata returned. Datasource may be private or restricted.")
json_data = datasources[0]


NameError: name 'response' is not defined

In [45]:
!pip uninstall langchain-tableau -y


Found existing installation: langchain_tableau 0.4.39
Uninstalling langchain_tableau-0.4.39:
  Successfully uninstalled langchain_tableau-0.4.39


In [46]:
!pip install --no-cache-dir langchain-tableau

In [47]:
pip show langchain-tableau


Name: langchain_tableau
Version: 0.4.39
Summary: Tableau tools for Agentic use cases with Langchain
Home-page: https://github.com/Tab-SE/tableau_langchain
Author: Stephen Price, Joe Constantino, Joseph Fluckiger
Author-email: 
License: MIT
Location: c:\Users\227457\Desktop\tableau_langchain_starter_kit-main\venv\Lib\site-packages
Requires: aiohttp, langchain, langchain-core, langchain-openai, langgraph, pydantic, pyjwt, python-dotenv, requests
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [48]:
from langchain_tableau.tools.query_tool import TableauDatasourceQueryTool


ModuleNotFoundError: No module named 'langchain_tableau.tools.query_tool'